In [17]:
import os
import sys
import csv
from pathlib import Path
import transformers
import warnings

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

# Use an absolute path to break the Jupyter "curse"
PROJECT_ROOT = Path(r"C:\Users\skali\OneDrive\Desktop\AI\AI test-task\task 1")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.config import load_config
from src.models.dictionary_ner import DictionaryNER
from src.models.crf_model import CRFModel
from src.models.bert_model import BERTModel
from src.llm.llm_judge import LocalLLMJudge

config_path = PROJECT_ROOT / "config.yaml"
config = load_config(config_path=str(config_path))

print(f"✅ Environment configured, imports succeeded!\n📂 Current directory: {Path.cwd()}")

✅ Environment configured, imports succeeded!
📂 Current directory: C:\Users\skali\OneDrive\Desktop\AI\AI test-task\task 1


In [18]:
dict_model = DictionaryNER(ignore_case=True)
csv_path = f"{config['paths']['data_processed']}/mountains.csv" 
with open(csv_path, "r", encoding="utf-8") as f:
    mountains = [row["name"] for row in csv.DictReader(f) if row.get("name")]
dict_model.load_dictionary(mountains)

# 2. CRF (Classic ML)
crf_model = CRFModel()
crf_model.load(f"{config['paths']['models_crf']}/crf_baseline.pkl")

# 3. Fine-tuned BERT (Deep Learning)
bert_model = BERTModel(model_path=f"{config['paths']['models_finetuned']}")

print("✅ All 3 models loaded into memory!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8675.10it/s]


✅ All 3 models loaded into memory!


In [19]:
test_text = "Last year I visited Nepal to see Mount Everest, but next time I want to conquer K2 and maybe a small hill near my house."
print(f"📝 Text for analysis: '{test_text}'\n")

models = [
    ("Dictionary (Exact match)", dict_model),
    ("CRF (Machine learning)", crf_model),
    ("Fine-Tuned BERT (Neural network)", bert_model)
]

bert_candidates = []

for name, model in models:
    print(f"--- {name} ---")
    result = model.predict(test_text)
    
    if isinstance(model, BERTModel):
        for entity in result:
            word = entity.get('word', '')
            score = entity.get('score', 0.0)
            bert_candidates.append(word)
            print(f"  🏔️ {word:<15} (Confidence: {score:.4f})")
    else:
        found_any = False
        for token, label in zip(result["tokens"], result["labels"]):
            if label != "O":
                print(f"  🏔️ {token:<15} {label}")
                found_any = True
        if not found_any:
             print("  (Nothing found)")
    print()

📝 Text for analysis: 'Last year I visited Nepal to see Mount Everest, but next time I want to conquer K2 and maybe a small hill near my house.'

--- Dictionary (Exact match) ---
  🏔️ Mount           B-MOUNTAIN
  🏔️ Everest         I-MOUNTAIN
  🏔️ K2              B-MOUNTAIN

--- CRF (Machine learning) ---
  🏔️ Mount           B-MOUNTAIN
  🏔️ Everest         I-MOUNTAIN

--- Fine-Tuned BERT (Neural network) ---
  🏔️ Mount           (Confidence: 1.0000)
  🏔️ K               (Confidence: 1.0000)
  🏔️ ##2             (Confidence: 0.9446)



In [20]:
print("🤖 Starting Phase 2: LLM Validation (Ollama Llama 3)")
print(f"BERT candidates: {bert_candidates}")

# Initialize the judge
judge = LocalLLMJudge(model_name="llama3")

# Filter the candidates
final_mountains = judge.filter_entities(bert_candidates)

print("\n✅ FINAL VERDICT (True mountains):")
for m in final_mountains:
    print(f"  ⭐ {m}")

🤖 Starting Phase 2: LLM Validation (Ollama Llama 3)
BERT candidates: ['Mount', 'K', '##2']

✅ FINAL VERDICT (True mountains):
  ⭐ K2
